[] Adjust Code-Gerüst, for channel interpolation: stick with the usual, checking continuous data and topoplots <br> 
[] Adjust topoplots for the fact that some subjects don't have retrieval events. If too complex, I'll use PSD instead, but I really think event-specific is better <br> 
[] Run script for subjects of different categories (behavioral exclusion, missing retrieval eeg data: 03, missing single behavioral trials: ...<br> 
[] Add comments <br> 

### Manual Quality Checks and Preprocessing Configuration

This notebook prepares the pre-processing steps that require manual input on a subject-level:<br> 
1. Checking the integrity of the recorded EEG data <br> 
2. Selection of channels for interpolation <br> 
3. Analyzing ICA results and selecting independent components for exclusion <br> 

The manual informations are logged in a pandas dataframe, that is used as input in the subsequent automatized preprocessing notebook (_3_preprocessing.ipynb_) which loops the preprocessing pipeline over a selection (or all) participants.


**1. Create entry in preprocessing_config.csv and checking for exclusion** <br>
This cell first checks whether the participant to-be-inspected should be included in the analyses according to their behavioral performance. It creates an entry in 'df_config' for each participant, allowing for manual decisions to be saved at the end of the notebook. If the participant is in the exclusion list, there's a warning and the rest of the notebook must not be run.

In [26]:
import mne
import pandas as pd
import json
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids
import matplotlib

# Sets the matplotlib backend to open a separate interactive window
%matplotlib qt 

# ==========================================
# 1. DEFINE CURRENT SUBJECT
# ==========================================
# Change this ID for every participant you want to inspect
subj = "03" 

# ==========================================
# 2. DEFINE PATHS & LOAD EXCLUSIONS DUE TO BEHAVIORAL RESULTS
# ==========================================
# This notebook should be located in project_folder/scripts/eeg
project_root = Path.cwd().parent.parent
bids_root = project_root / "data" / "bids"
derivatives_dir = project_root / "data" / "derivatives"

# Load json object with subjects that should be entirely excluded from the analysis
exclusion_file = derivatives_dir / "subject_exclusions.json"

with open(exclusion_file, "r") as f:
    exclusions = json.load(f)

# Use .get() to safely load the list, defaulting to an empty list if the key is missing
behavioral_exclusions = exclusions.get("behavioral_exclusions", [])

print("The following subjects are marked for exclusion based on behavioral criteria:")
print(behavioral_exclusions)

# Path to the central configuration file
config_path = derivatives_dir / "preprocessing_config.csv"

# ==========================================
# 3. LOAD OR CREATE CONFIGURATION FILE
# ==========================================
if config_path.exists():
    df_config = pd.read_csv(config_path, dtype={'subject': str})
    print(f"\nExisting config loaded. {len(df_config)} subjects inspected so far.")
else:
    df_config = pd.DataFrame(columns=[
        "subject", 
        "is_excluded", 
        "eeg_enc_recorded", 
        "eeg_ret_recorded", 
        "beh_enc_complete", 
        "beh_ret_complete",
        "bad_channels", 
        "bad_icas", 
        
        "notes_exclusion", 
        "notes_beh", 
        "notes_eeg"
    ])
    print("\nNo existing config found. Creating a new detailed metadata table.")

# ==========================================
# 4. SAFETY CHECK & AUTO-LOGGING
# ==========================================
# Check if the current subject is in the exclusion list
subject_is_excluded = subj in behavioral_exclusions

if subject_is_excluded:
    # 4a. Throw a highly visible warning
    print("\n" + "!"*65)
    print(f" WARNING: Subject {subj} is in the behavioral_exclusions list!")
    print("!"*65)
    
    # 4b. Automatically log this into the config dataframe
    # NOTE: since ICA/channel inspection never runs for auto-excluded subjects,
    # bad_channels/bad_icas/notes_eeg are left empty here rather than
    # guessed at — nothing in this branch touches EEG data at all.
    auto_notes = "Auto-excluded: Behavioral criteria not met."
    
    if subj in df_config['subject'].values:
        idx = df_config.index[df_config['subject'] == subj].tolist()[0]
        df_config.loc[idx, 'is_excluded'] = True
        df_config.loc[idx, 'notes_exclusion'] = auto_notes
    else:
        new_row = pd.DataFrame([{
            "subject": subj, 
            "is_excluded": True,
            "eeg_enc_recorded": pd.NA,
            "eeg_ret_recorded": pd.NA,
            "beh_enc_complete": pd.NA,
            "beh_ret_complete": pd.NA,
            "bad_channels": "", 
            "bad_icas": "",
            "ica_notes": "",
            "notes_exclusion": auto_notes,
            "notes_beh": "",
            "notes_eeg": ""
        }])
        df_config = pd.concat([df_config, new_row], ignore_index=True)
        
    # 4c. Save immediately so you don't need to run the final cell
    df_config.to_csv(config_path, index=False)
    print(f"\n--> Action taken: Subject {subj} automatically marked as excluded in the config.")
    print("--> You can SKIP the rest of the cells in this notebook for this subject.")

else:
    print(f"\n--> Subject {subj} is clear for preprocessing. Proceed to the next cells.")

The following subjects are marked for exclusion based on behavioral criteria:
['05', '06', '07', '08', '13', '14', '16', '18', '19', '24', '25', '28', '31', '34', '39', '40']

No existing config found. Creating a new detailed metadata table.

--> Subject 03 is clear for preprocessing. Proceed to the next cells.


**Event marker integrity**: Creates plot to check if all event triggers were recorded as expected.

In [27]:
# ==========================================
# 4. LOAD BIDS DATA
# ==========================================
bids_path = BIDSPath(
    subject=subj, 
    task='loc', 
    datatype='eeg', 
    root=bids_root)

# 2. Load Raw
raw_eeg, bids_event_id = read_raw_bids(
    bids_path=bids_path, 
    return_event_dict=True,  # Extracts events mapping from BIDS (previous botebook)
    verbose='error'
)
raw_eeg.load_data()

# 3. Load events
events, event_id = mne.events_from_annotations(
    raw_eeg, 
    event_id=bids_event_id,  
    verbose=False
)

# 4. Plot event markers as function of experiment run time
fig_ev = mne.viz.plot_events(
    events, 
    sfreq=raw_eeg.info["sfreq"], 
    first_samp=raw_eeg.first_samp, 
    event_id=event_id
)

Reading 0 ... 2982939  =      0.000 ...  2982.939 secs...


C:\Users\noahm\AppData\Local\Temp\ipykernel_12464\2064455744.py:26: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  fig_ev = mne.viz.plot_events(


**Continuous plot**: catches transient issues — sudden jumps, drift, popping, or intermittent flat segments — that a PSD averaged over the whole recording could smear out or dilute.

In [28]:
# ==========================================
# 5. VISUAL INSPECTION (CONTINUOUS DATA)
# ==========================================
print(f"Inspecting subject: {subj}")

# Create a copy with basic filters for easier visual inspection
# (1Hz highpass removes slow drifts, 50/60Hz notch removes line noise)
raw_inspect = raw_eeg.copy().filter(l_freq=1.0, h_freq=None).notch_filter(50) 

# Step 5a: Plot time series 
# (Click on a channel name in the plot to mark it as 'bad'. Press 'b' for butterfly view)
raw_inspect.plot(
    duration=10, 
    n_channels=32, 
    scalings=dict(eeg=20e-6),
    theme="light" 
)

# MNE remembers which channels you clicked:
# In the study log book, we already marked bad channels. Here, I verify again and mark them by clicking.
print(f"Bad channels marked during inspection: {raw_inspect.info['bads']}")

Inspecting subject: 03
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Fil

Channels marked as bad:
none


**PSD Plot**: catches channels with abnormal spectral signatures — e.g., a channel with a flat/near-zero PSD across all frequencies (dead/disconnected), a channel with excess broadband power (poor contact, high impedance), or a channel with an unusual line-noise peak that the notch didn't fully suppress (bad shielding/contact at that specific electrode).

**Flat/dead**: near-zero variance in the continuous plot and a flat PSD across all frequencies → interpolate. <br>
**Pure line noise residual**: a sharp peak at 50 Hz (or 100 Hz) in the PSD, isolated to that channel, not shared by neighbors → interpolate rather than trying to fix with a narrower notch (which risks distorting other channels).<br>
**Broadband noisy**: elevated power across most/all frequencies relative to neighboring channels of similar cortical location → interpolate. <br>
**Spatial outlier in topoplot**: consistently an isolated extreme value (hot or cold) across multiple epochs/conditions, not just one epoch → interpolate.<br>
**Borderline/ambiguous**: if a channel only shows a mild issue in one method and looks fine in the other two, I'd lean toward not interpolating and letting ICA + later epoch-level rejection (Autoreject) handle it — over-interpolating loses real signal and inflates spatial correlation between channels, which matters for your cluster-based permutation tests.<br>

In [29]:
# 2. Run your plot command
fig = raw_inspect.compute_psd(fmax=100).plot()

Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).


**Averaged topoplots over epochs**: catches spatial outliers — a channel that stands out as an isolated hot/cold spot relative to its neighbors, which is often more obvious spatially than in a raw time series or single-channel PSD, especially for subtle "bridging" or "borderline noisy" channels that don't jump out otherwise.



In [30]:
# 2. Define both events as list of tuples: (name, id)
target_events = [
    ('enc_fixation', 110),
    ('ret_fixation', 210),
]

for event_name, event_id in target_events:

    # 3. Extract events for this specific event only
    events, _ = mne.events_from_annotations(
        raw_inspect,
        event_id={event_name: event_id},
        verbose=False
    )

    # 4. Check if the target ID exists
    if event_id not in events[:, 2]:
        print(f"Warning: Event ID {event_id} ('{event_name}') not found. Skipping.")
        continue

    print(f"Plotting topomap for event: '{event_name}' (ID: {event_id})")

    # 5. Create Epochs
    epochs_viz = mne.Epochs(
        raw_inspect,
        events,
        event_id={event_name: event_id},
        tmin=-0.5, tmax=1,
        baseline=(None, 0),
        preload=True,
        reject_by_annotation=True,
        verbose=False
    )

    # 6. Plot Topomap
    #    Look for "Red" or "Blue" spots that look completely different from neighbors
    fig_topo = epochs_viz.average().plot_topomap(
        show_names=True,
        size=3,
        nrows = 1  # Zur Unterscheidung
    )

Plotting topomap for event: 'enc_fixation' (ID: 110)


ValueError: Could not find any of the events you specified.

In [31]:
# ==========================================
# 6. RUN INDEPENDENT COMPONENT ANALYSIS (ICA)
# ==========================================
# Transfer bad channels from the visual inspection so ICA ignores them
raw_eeg.info['bads'] = ['O1', 'CP5', 'C3', 'T8']

# Create a new copy for ICA (1Hz highpass is highly recommended for ICA)
raw_ica = raw_eeg.copy().filter(l_freq=1.0, h_freq=None)

# Initialize and fit ICA
from mne.preprocessing import ICA
ica = ICA(n_components=15, random_state=97, max_iter='auto')
ica.fit(raw_ica)

# ==========================================
# 7. AUTOMATED EOG DETECTION & VISUALIZATION
# ==========================================
# Let MNE suggest EOG components based on your dedicated EOG channels
eog_indices, eog_scores = ica.find_bads_eog(raw_ica)
print(f"MNE automatically identified these ICs as EOG artifacts: {eog_indices}")

# Plot components (Topographies)
ica.plot_components()

# Plot ICA sources over time (Time series)
# You can click on the IC name here to mark it as bad, but we will explicitly
# write down any ADDITIONAL bad ICs in the next cell to save them.
# ica.plot_sources(raw_ica, block=True)

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 3301 samples (3.301 s)



KeyboardInterrupt: 

In [32]:
# ==========================================
# 6. RUN INDEPENDENT COMPONENT ANALYSIS (ICA)
# ==========================================
import matplotlib
matplotlib.use('Agg')  # No interactive windows — just save figures to disk
import matplotlib.pyplot as plt

raw_eeg.info['bads'] = ['O1', 'CP5', 'C3', 'T8']

raw_ica = raw_eeg.copy().filter(l_freq=1.0, h_freq=None)
raw_ica.set_eeg_reference('average')
raw_ica.resample(100)  # Downsample for ICA speed — safe for ICLabel purposes

rank = mne.compute_rank(raw_ica, tol='auto')['eeg']
print(f"Using n_components={rank} based on computed data rank.")

from mne.preprocessing import ICA
ica = ICA(
    n_components=rank,
    method='picard',
    fit_params=dict(ortho=False, extended=True),  # Picard config matching extended Infomax behavior
    random_state=97,
    max_iter='auto'
)
ica.fit(raw_ica, decim=3)

# ==========================================
# 7. EOG DETECTION + ICLABEL CLASSIFICATION
# ==========================================
eog_indices, eog_scores = ica.find_bads_eog(raw_ica)

from mne_icalabel import label_components
ic_labels = label_components(raw_ica, ica, method='iclabel')
labels = ic_labels['labels']
probs = ic_labels['y_pred_proba']

# Auto-exclude: any non-brain, non-other label with prob > 0.90, plus EOG-detected
auto_exclude = sorted(set(eog_indices) | {
    idx for idx, (label, prob) in enumerate(zip(labels, probs))
    if label not in ('brain', 'other') and prob > 0.90
})

print(f"\n--- Auto-excluded ICs (>90% confidence or EOG-detected) ---")
for idx in auto_exclude:
    print(f"IC{idx:02d}: {labels[idx]} (p={probs[idx]:.2f})")

# Borderline candidates you're explicitly choosing NOT to look at now,
# but flagging in case something feels off later
borderline = [
    idx for idx, (label, prob) in enumerate(zip(labels, probs))
    if label not in ('brain', 'other', ) and 0.70 < prob <= 0.90 and idx not in auto_exclude
]
if borderline:
    print(f"Borderline (70-90%, not auto-excluded): {[(idx, labels[idx], round(probs[idx],2)) for idx in borderline]}")

confirmed_exclude_icas = auto_exclude

# ==========================================
# 7b. SAVE DIAGNOSTIC PLOTS TO DISK (no display)
# ==========================================
import os
subj_deriv_dir = os.path.join(derivatives_dir, f"sub-{subj}", "eeg", "ica_qc")
os.makedirs(subj_deriv_dir, exist_ok=True)

# Overview of all components
fig_components = ica.plot_components(show=False)
if isinstance(fig_components, list):
    for i, f in enumerate(fig_components):
        f.savefig(os.path.join(subj_deriv_dir, f"components_overview_{i}.png"), dpi=100)
        plt.close(f)
else:
    fig_components.savefig(os.path.join(subj_deriv_dir, "components_overview.png"), dpi=100)
    plt.close(fig_components)

# Full diagnostic plots ONLY for excluded components (not all 15+)
if confirmed_exclude_icas:
    figs = ica.plot_properties(raw_ica, picks=confirmed_exclude_icas, show=False)
    for idx, fig in zip(confirmed_exclude_icas, figs):
        fname = f"IC{idx:02d}_{labels[idx]}_{probs[idx]:.2f}.png"
        fig.savefig(os.path.join(subj_deriv_dir, fname), dpi=100)
        plt.close(fig)

print(f"\nSaved QC plots to: {subj_deriv_dir}")

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 3301 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Computing rank from data with rank=None
    Using tolerance 6.1e-11 (2.2e-16 eps * 59 dim * 4.6e+03  max singular value)
    Estimated rank (eeg): 58
    EEG: rank 58 computed from 59 data channels with 0 projectors
Using n_components=58 based on computed data rank.
Fitting ICA to data using 59 channels (please be patient, this may take a while)
Selecting by number: 58 components
Fitting ICA took 80.7s.
Using EOG channels: HEOG, VEOG
..

C:\Users\noahm\AppData\Local\Temp\ipykernel_12464\757422567.py:33: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_ica, ica, method='iclabel')



--- Auto-excluded ICs (>90% confidence or EOG-detected) ---
IC00: eye blink (p=0.99)
IC02: eye blink (p=0.93)
IC03: eye blink (p=1.00)
IC04: eye blink (p=0.95)
IC15: muscle artifact (p=0.92)
IC19: muscle artifact (p=0.96)
IC26: muscle artifact (p=0.99)
IC34: muscle artifact (p=0.93)
IC37: muscle artifact (p=1.00)
IC38: muscle artifact (p=0.93)
Borderline (70-90%, not auto-excluded): [(14, 'muscle artifact', np.float32(0.73)), (41, 'muscle artifact', np.float32(0.81)), (44, 'muscle artifact', np.float32(0.75)), (45, 'muscle artifact', np.float32(0.77)), (49, 'muscle artifact', np.float32(0.72)), (54, 'channel noise', np.float32(0.78))]
    Using multitaper spectrum estimation with 7 DPSS windows
Not setting metadata
1491 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
1491 matching events found
No baseline correction applied
0 projection items activated
Not setting metadata
1491 matching events found
No baseline correction applied


In [34]:
# ==========================================
# 8. LOG YOUR MANUAL DECISIONS & METADATA
# ==========================================
# Check if the subject is in your predefined full-exclusion list
subject_is_excluded = subj in behavioral_exclusions

# A. Phase Completion Flags (Booleans)
# Change these to False if a specific phase is missing or corrupted
my_eeg_enc_recorded = True
my_eeg_ret_recorded = False
my_beh_enc_complete = True
my_beh_ret_complete = True

# B. Bad Channels & ICAs
my_bad_channels = raw_eeg.info['bads'] if not subject_is_excluded else []
# my_manual_icas = confirmed_exclude_icas if not subject_is_excluded else []

# B2. Build a short ICLabel justification string for each excluded IC
ica_reasons = ", ".join(
    f"IC{idx:02d}:{labels[idx]}({probs[idx]:.2f})"
    for idx in confirmed_exclude_icas
) if confirmed_exclude_icas else ""

# C. Targeted Notes
my_notes_exclusion = ""
my_notes_beh = ""
my_notes_eeg = "No triggers were recorded during retrieval"
my_notes_ICA = f"ICA excluded: {ica_reasons}" if ica_reasons else ""

# ==========================================
# 9. FORMAT AND SAVE TO CSV
# ==========================================
bads_str = ", ".join(my_bad_channels) if my_bad_channels else ""
icas_str = ", ".join(map(str, confirmed_exclude_icas)) if confirmed_exclude_icas else ""

# Prepare the dictionary for the current subject
row_data = {
    "subject": subj, 
    "is_excluded": subject_is_excluded,
    "eeg_enc_recorded": my_eeg_enc_recorded,
    "eeg_ret_recorded": my_eeg_ret_recorded,
    "beh_enc_complete": my_beh_enc_complete,
    "beh_ret_complete": my_beh_ret_complete,
    "bad_channels": bads_str, 
    "bad_icas": icas_str,
    "ica_notes": my_notes_ICA,
    "notes_exclusion": my_notes_exclusion,
    "notes_beh": my_notes_beh,
    "notes_eeg": my_notes_eeg
}

if subj in df_config['subject'].values:
    # Update the existing row using the index
    idx = df_config.index[df_config['subject'] == subj].tolist()[0]
    for key, value in row_data.items():
        df_config.loc[idx, key] = value
    print(f"--> Metadata for sub-{subj} successfully updated.")
else:
    # Append a new row
    new_row = pd.DataFrame([row_data])
    df_config = pd.concat([df_config, new_row], ignore_index=True)
    print(f"--> New metadata entry for sub-{subj} successfully added.")

df_config.to_csv(config_path, index=False)
display(df_config.tail())

--> Metadata for sub-03 successfully updated.


,subject,is_excluded,eeg_enc_recorded,eeg_ret_recorded,beh_enc_complete,beh_ret_complete,bad_channels,bad_icas,notes_exclusion,notes_beh,notes_eeg,ica_notes
0,03,False,True,False,True,True,"O1, CP5, C3, T8","0, 2, 3, 4, 15, 19, 26, 34, 37, 38",,,No triggers were recorded during retrieval,"ICA excluded: IC00:eye blink(0.99), IC02:eye b..."
